### Imports

In [56]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

### Setup

In [57]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "Data"
OUT_DIR  = BASE_DIR / "Data_standard"
OUT_DIR.mkdir(parents=True, exist_ok=True)

read  = lambda f: pd.read_csv(DATA_DIR / f, dtype=str)         # text keeps leading zeros
num   = lambda s: pd.to_numeric(s, errors="coerce")
iso   = lambda s: pd.to_datetime(s, format="%m/%d/%Y", errors="coerce").dt.strftime("%Y-%m-%d")


### Readable Names

In [58]:
HOSP_CD = {
 'COMP_HIP_KNEE':('complications_hip_knee_replacement','lower'),'Hybrid_HWM':('mortality_hospital_wide_all_cause','lower'),
 'MORT_30_AMI':('mortality_30day_heart_attack','lower'),'MORT_30_CABG':('mortality_30day_cabg_surgery','lower'),
 'MORT_30_COPD':('mortality_30day_copd','lower'),'MORT_30_HF':('mortality_30day_heart_failure','lower'),
 'MORT_30_PN':('mortality_30day_pneumonia','lower'),'MORT_30_STK':('mortality_30day_stroke','lower'),
 'PSI_03':('psi03_pressure_ulcer_rate','lower'),'PSI_04':('psi04_death_after_serious_surgical_complication','lower'),
 'PSI_06':('psi06_iatrogenic_pneumothorax_rate','lower'),'PSI_08':('psi08_inhospital_fall_fracture_rate','lower'),
 'PSI_09':('psi09_postop_hemorrhage_hematoma_rate','lower'),'PSI_10':('psi10_postop_kidney_injury_dialysis_rate','lower'),
 'PSI_11':('psi11_postop_respiratory_failure_rate','lower'),'PSI_12':('psi12_postop_pulmonary_embolism_dvt_rate','lower'),
 'PSI_13':('psi13_postop_sepsis_rate','lower'),'PSI_14':('psi14_postop_wound_dehiscence_rate','lower'),
 'PSI_15':('psi15_accidental_puncture_laceration_rate','lower'),'PSI_90':('psi90_patient_safety_composite','lower')}
HOSP_TE = {
 'OP_31':('cataract_visual_function_improved_pct','higher'),'OP_29':('colonoscopy_followup_interval_appropriate_pct','higher'),
 'GMCS':('malnutrition_composite_score','higher'),'GMCS_Malnutrition_Diagnosis_Documented':('malnutrition_diagnosis_documented','higher'),
 'GMCS_Malnutrition_Screening':('malnutrition_risk_screening','higher'),'GMCS_Nutrition_Assessment':('malnutrition_nutrition_assessment','higher'),
 'GMCS_Nutritional_Care_Plan':('malnutrition_nutritional_care_plan','higher'),
 'HH_HYPER':('harm_severe_hyperglycemia','lower'),'HH_HYPO':('harm_severe_hypoglycemia','lower'),'HH_ORAE':('harm_opioid_adverse_events','lower'),
 'OP_40':('stemi_measure_op40','higher'),'SAFE_USE_OF_OPIOIDS':('opioid_concurrent_prescribing','lower'),
 'STK_02':('stroke_antithrombotic_at_discharge','higher'),'STK_03':('stroke_anticoagulation_afib','higher'),
 'STK_05':('stroke_antithrombotic_by_day2','higher'),'VTE_1':('vte_prophylaxis','higher'),'VTE_2':('icu_vte_prophylaxis','higher'),
 'EDV':('ed_volume_category','n/a'),
 'OP_18a':('ed_median_minutes_all_patients','lower'),'OP_18b':('ed_median_minutes_excl_transfers_psych','lower'),
 'OP_18c':('ed_median_minutes_psych_patients','lower'),'OP_18d':('ed_median_minutes_transferred_patients','lower'),
 'OP_22':('ed_left_before_seen_pct','lower'),'OP_23':('ed_head_ct_results_timely_pct','higher'),
 'IMM_3':('staff_flu_vaccination_pct','higher'),'SEP_1':('sepsis_care_bundle_pct','higher'),
 'SEP_SH_3HR':('septic_shock_3hr_bundle','higher'),'SEP_SH_6HR':('septic_shock_6hr_bundle','higher'),
 'SEV_SEP_3HR':('severe_sepsis_3hr_bundle','higher'),'SEV_SEP_6HR':('severe_sepsis_6hr_bundle','higher')}
# ASC: id -> (short name, direction, official name from CMS dictionary, data period start, end)
ASC = {
 'ASC_1':('patient_burn','lower','Number of patients who experience a burn prior to discharge from the ASC','2024-01-01','2024-12-31'),
 'ASC_2':('patient_fall','lower','Number of patients who experience a fall within the ASC','2024-01-01','2024-12-31'),
 'ASC_3':('wrong_site_side_patient_procedure_implant','lower','Number of patients who experience a wrong site, side, patient, procedure, or implant','2024-01-01','2024-12-31'),
 'ASC_4':('hospital_transfer_admission','lower','Percentage of ASC patients who are transferred or admitted to a hospital upon discharge from the ASC','2024-01-01','2024-12-31'),
 'ASC_9':('colonoscopy_followup_interval_appropriate','higher','Endoscopy/Polyp Surveillance: Appropriate Follow-up Interval for Normal Colonoscopy in Average Risk Patients','2024-01-01','2024-12-31'),
 'ASC_11':('cataract_visual_function_improved','higher',"Improvement in Patient's Visual Function within 90 Days Following Cataract Surgery",'2024-01-01','2024-12-31'),
 'ASC_12':('hospital_visit_after_colonoscopy_7day','lower','Facility 7-Day Risk Standardized Hospital Visit Rate after Outpatient Colonoscopy','2022-01-01','2024-12-31'),
 'ASC_13':('normothermia_outcome','higher','Normothermia (body temperature of 96.8 F or higher within 15 minutes of arriving in the post-anesthesia care unit)','2024-01-01','2024-12-31'),
 'ASC_14':('unplanned_anterior_vitrectomy','lower','Unplanned Anterior Vitrectomy (percentage of cataract surgeries with unplanned additional eye surgery)','2024-01-01','2024-12-31'),
 'ASC_17':('hospital_visit_after_orthopedic_procedure','lower','Hospital Visits after Orthopedic Ambulatory Surgical Center Procedures','2023-01-01','2024-12-31'),
 'ASC_18':('hospital_visit_after_urology_procedure','lower','Hospital Visits after Urology Ambulatory Surgical Center Procedures','2023-01-01','2024-12-31'),
 'ASC_19':('hospital_visit_after_general_surgery','lower','Hospital Visits after General Surgery Procedures Performed','2023-01-01','2024-12-31')}
RSHV = ['ASC_12','ASC_17','ASC_18','ASC_19']    # ASC measures reported as risk-standardized rate + interval + category
# ASC periods come from the Measure Dates table in the CMS data dictionary (July 2026); the ASC files themselves carry only 'Year'.

In [59]:
def std_vs_national(s):
    """Unify wording: Better / No Different / Worse / Too Few Cases / Not Available."""
    s = s.fillna("Not Available").str.strip()
    out = pd.Series("Not Available", index=s.index)
    out[s.str.startswith("Better")] = "Better"
    out[s.str.startswith("No Different")] = "No Different"
    out[s.str.startswith("Worse")] = "Worse"
    out[s.str.contains("Too Small|Too Few", case=False)] = "Too Few Cases"
    return out

def split_score(raw):
    """raw text -> (numeric score, free text such as 'very high', status)."""
    n = num(raw)
    na = raw.isna() | raw.str.strip().str.lower().isin(["not available", "n/a"])
    text = raw.where(n.isna() & ~na)
    status = np.where(n.notna(), "reported", np.where(na, "Not Available", "reported"))
    return n, text, status

FACILITY_COLS = ['geography_level','facility_type','source_file','facility_id','npi','shared_facility_id','facility_name','address','city','state','zip',
  'county','phone','measure_group','condition','measure_id','measure_short_name','measure_name','better_direction','score','score_text','score_status',
  'vs_national','cases','ci_lower','ci_upper','footnote_code','period_start','period_end']
STATE_COLS = ['geography_level','facility_type','source_file','state','measure_group','condition','measure_id','measure_short_name','measure_name',
  'better_direction','ed_volume_tier','statistic','value','value_status','footnote_code','period_start','period_end']

In [60]:
raw = read("Complications_and_Deaths-Hospital.csv")
sc, tx, st = split_score(raw["Score"])
fac_cd = pd.DataFrame({
  'geography_level':'facility','facility_type':'hospital','source_file':'Complications_and_Deaths-Hospital.csv',
  'facility_id':raw['Facility ID'],'npi':None,'shared_facility_id':False,'facility_name':raw['Facility Name'],'address':raw['Address'],
  'city':raw['City/Town'],'state':raw['State'],'zip':raw['ZIP Code'],'county':raw['County/Parish'],'phone':raw['Telephone Number'],
  'measure_group':'Complications and Deaths','condition':None,'measure_id':raw['Measure ID'],
  'measure_short_name':raw['Measure ID'].map(lambda m: HOSP_CD[m][0]),'measure_name':raw['Measure Name'],
  'better_direction':raw['Measure ID'].map(lambda m: HOSP_CD[m][1]),'score':sc,'score_text':tx,'score_status':st,
  'vs_national':std_vs_national(raw['Compared to National']),'cases':num(raw['Denominator']),
  'ci_lower':num(raw['Lower Estimate']),'ci_upper':num(raw['Higher Estimate']),'footnote_code':raw['Footnote'],
  'period_start':iso(raw['Start Date']),'period_end':iso(raw['End Date'])})[FACILITY_COLS]
print(fac_cd.shape)

(95800, 29)


In [61]:
raw = read("Timely_and_Effective_Care-Hospital.csv")
sc, tx, st = split_score(raw["Score"])
fac_te = pd.DataFrame({
  'geography_level':'facility','facility_type':'hospital','source_file':'Timely_and_Effective_Care-Hospital.csv',
  'facility_id':raw['Facility ID'],'npi':None,'shared_facility_id':False,'facility_name':raw['Facility Name'],'address':raw['Address'],
  'city':raw['City/Town'],'state':raw['State'],'zip':raw['ZIP Code'],'county':raw['County/Parish'],'phone':raw['Telephone Number'],
  'measure_group':'Timely and Effective Care','condition':raw['Condition'],'measure_id':raw['Measure ID'],
  'measure_short_name':raw['Measure ID'].map(lambda m: HOSP_TE[m][0]),'measure_name':raw['Measure Name'],
  'better_direction':raw['Measure ID'].map(lambda m: HOSP_TE[m][1]),'score':sc,'score_text':tx,'score_status':st,
  'vs_national':'Not Available','cases':num(raw['Sample']),'ci_lower':np.nan,'ci_upper':np.nan,'footnote_code':raw['Footnote'],
  'period_start':iso(raw['Start Date']),'period_end':iso(raw['End Date'])})[FACILITY_COLS]
print(fac_te.shape)

(138084, 29)


In [62]:
asc = read("ASC_Facility.csv")
asc['shared_facility_id'] = asc.duplicated('Facility ID', keep=False)       # same ID, different NPI/name
parts = []
for mid, (short, direction, official, p0, p1) in ASC.items():
    k = mid.replace('_', '-')                                               # raw prefix, e.g. 'ASC-12'
    if mid in RSHV:
        score_raw = asc[f'{k} RSHV Rate']; cases = num(asc[f'{k} Total Cases'])
        vs = std_vs_national(asc[f'{k} Performance Category'])
        lo, hi = num(asc[f'{k} Interval Lower Limit']), num(asc[f'{k} Interval Upper Limit'])
    else:
        score_raw = asc[f'{k} Rate*']; cases = np.nan; vs = 'Not Available'; lo = hi = np.nan
    sc, tx, stt = split_score(score_raw)
    parts.append(pd.DataFrame({
      'geography_level':'facility','facility_type':'asc','source_file':'ASC_Facility.csv','facility_id':asc['Facility ID'],'npi':asc['NPI'],
      'shared_facility_id':asc['shared_facility_id'],'facility_name':asc['Facility Name'],'address':None,'city':asc['City/Town'],
      'state':asc['State'],'zip':asc['ZIP Code'],'county':None,'phone':None,'measure_group':'ASC Quality Reporting','condition':None,
      'measure_id':mid,'measure_short_name':short,'measure_name':official,'better_direction':direction,'score':sc,'score_text':tx,
      'score_status':stt,'vs_national':vs,'cases':cases,'ci_lower':lo,'ci_upper':hi,'footnote_code':asc[f'{k} Footnote'],
      'period_start':p0,'period_end':p1}))
fac_asc = pd.concat(parts, ignore_index=True)[FACILITY_COLS]
print(fac_asc.shape)

(68532, 29)


In [63]:
import re
def state_vs(df, **fixed):
    for k, v in fixed.items(): df[k] = v
    return df

# --- Complications & deaths (state): counts of hospitals ---
raw = read("Complications_and_Deaths-State.csv")
cnt = {'Number of Hospitals Better':'n_better','Number of Hospitals Same':'n_same','Number of Hospitals Worse':'n_worse','Number of Hospitals Too Few':'n_too_few'}
long = raw.melt(id_vars=['State','Measure ID','Measure Name','Footnote','Start Date','End Date'], value_vars=list(cnt), var_name='statistic', value_name='raw')
long['statistic'] = long['statistic'].map(cnt)
val = num(long['raw'])
st_cd = pd.DataFrame({'geography_level':'state','facility_type':'hospital','source_file':'Complications_and_Deaths-State.csv','state':long['State'],
  'measure_group':'Complications and Deaths','condition':None,'measure_id':long['Measure ID'],
  'measure_short_name':long['Measure ID'].map(lambda m: HOSP_CD[m][0]),'measure_name':long['Measure Name'],
  'better_direction':long['Measure ID'].map(lambda m: HOSP_CD[m][1]),'ed_volume_tier':None,'statistic':long['statistic'],'value':val,
  'value_status':np.where(val.notna(),'reported',long['raw'].fillna('Not Available')),'footnote_code':long['Footnote'],
  'period_start':iso(long['Start Date']),'period_end':iso(long['End Date'])})[STATE_COLS]

# --- Timely & effective care (state): one value per state x measure (ED times also by ED-volume tier) ---
raw = read("Timely_and_Effective_Care-State.csv")
tier = raw['Measure ID'].str.extract(r'_(VERY_HIGH|HIGH|MEDIUM|LOW)_MIN$')[0].str.lower().str.replace('_', ' ')
base = raw['Measure ID'].str.replace(r'_(VERY_HIGH|HIGH|MEDIUM|LOW)_MIN$', '', regex=True)
val = num(raw['Score'])
st_te = pd.DataFrame({'geography_level':'state','facility_type':'hospital','source_file':'Timely_and_Effective_Care-State.csv','state':raw['State'],
  'measure_group':'Timely and Effective Care','condition':raw['Condition'],'measure_id':base,
  'measure_short_name':base.map(lambda m: HOSP_TE[m][0]),'measure_name':raw['Measure Name'],
  'better_direction':base.map(lambda m: HOSP_TE[m][1]),'ed_volume_tier':tier,'statistic':'score','value':val,
  'value_status':np.where(val.notna(),'reported',raw['Score'].fillna('Not Available')),'footnote_code':raw['Footnote'],
  'period_start':iso(raw['Start Date']),'period_end':iso(raw['End Date'])})[STATE_COLS]

# --- ASC (state): average/median facility rate, or counts of ASCs better/same/worse/too small ---
raw = read("ASC_State.csv"); rows = []
catmap = {'Better':'n_better','No Different':'n_same','Worse':'n_worse','Too Small':'n_too_few'}
for col in raw.columns:
    if col in ('State', 'Year'): continue
    m = re.match(r'(?:(Avg|Median) )?(ASC-\d+) (?:State Rate\*|(Better|No Different|Worse|Too Small))$', col)
    stat, k, cat = m.groups(); mid = k.replace('-', '_')
    statistic = {'Avg':'avg','Median':'median'}[stat] if stat else catmap[cat]
    v = num(raw[col]); short, direction, official, p0, p1 = ASC[mid]
    rows.append(pd.DataFrame({'geography_level':'state','facility_type':'asc','source_file':'ASC_State.csv','state':raw['State'],
      'measure_group':'ASC Quality Reporting','condition':None,'measure_id':mid,'measure_short_name':short,'measure_name':official,
      'better_direction':direction,'ed_volume_tier':None,'statistic':statistic,'value':v,
      'value_status':np.where(v.notna(),'reported','Not Available'),                       # e.g. source text "N/A (5)"
      'footnote_code':raw[col].str.extract(r'\((\d+)\)')[0],'period_start':p0,'period_end':p1}))
st_asc = pd.concat(rows, ignore_index=True)[STATE_COLS]
print(st_cd.shape, st_te.shape, st_asc.shape)

(4480, 17) (1736, 17) (1728, 17)


In [64]:
raw_cd, raw_te, raw_asc = read("Complications_and_Deaths-Hospital.csv"), read("Timely_and_Effective_Care-Hospital.csv"), read("ASC_Facility.csv")
assert len(fac_cd) == len(raw_cd) and len(fac_te) == len(raw_te) and len(fac_asc) == len(raw_asc) * len(ASC)
assert fac_cd.score.notna().sum() == num(raw_cd.Score).notna().sum() and fac_te.score.notna().sum() == num(raw_te.Score).notna().sum()
assert len(st_te) == len(read("Timely_and_Effective_Care-State.csv")) and len(st_cd) == len(read("Complications_and_Deaths-State.csv")) * 4
assert list(fac_cd.columns) == list(fac_te.columns) == list(fac_asc.columns) == FACILITY_COLS
assert list(st_cd.columns) == list(st_te.columns) == list(st_asc.columns) == STATE_COLS
assert all(f.measure_short_name.notna().all() for f in (fac_cd, fac_te, fac_asc, st_cd, st_te, st_asc))

outputs = {'facility_hospital_complications_deaths':fac_cd, 'facility_hospital_timely_effective_care':fac_te, 'facility_asc':fac_asc,
           'state_hospital_complications_deaths':st_cd, 'state_hospital_timely_effective_care':st_te, 'state_asc':st_asc}
for name, df in outputs.items():
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)
    print(f"{name:45s} {len(df):>8,} rows x {df.shape[1]} cols")

facility_hospital_complications_deaths          95,800 rows x 29 cols
facility_hospital_timely_effective_care        138,084 rows x 29 cols
facility_asc                                    68,532 rows x 29 cols
state_hospital_complications_deaths              4,480 rows x 17 cols
state_hospital_timely_effective_care             1,736 rows x 17 cols
state_asc                                        1,728 rows x 17 cols


In [ ]:
DESC = {
 'geography_level':'facility or state',
 'facility_type':'hospital or asc: what kind of facility the row is about / summarizes',
 'source_file':'original CSV the row came from',
 'facility_id':'CMS facility ID (hospital CCN is 6 chars; ASC ID is 10 chars; different systems)',
 'npi':'National Provider Identifier (ASC rows only)',
 'shared_facility_id':'True if the same facility_id appears on more than one ASC row',
 'facility_name':'facility name',
 'address':'street address (hospitals only)',
 'city':'city/town',
 'state':'2-letter state/territory code',
 'zip':'ZIP code (text)',
 'county':'county/parish (hospitals only)',
 'phone':'telephone (hospitals only)',
 'measure_group':'CMS program the measure belongs to',
 'condition':'CMS condition group (timely care only)',
 'measure_id':'CMS measure ID',
 'measure_short_name':'readable snake_case measure name (added)',
 'measure_name':'measure name as published by CMS',
 'better_direction':'"lower", "higher" or "n/a": which direction is better',
 'score':'numeric measure value; blank if not reported',
 'score_text':'non-numeric score text, e.g. ED volume "very high"',
 'score_status':'reported" or "Not Available',
 'vs_national':'Better / No Different / Worse / Too Few Cases / Not Available (only some measures have it)',
 'cases':'denominator or sample size or total cases, whichever the source provides',
 'ci_lower':'lower interval limit',
 'ci_upper':'upper interval limit',
 'footnote_code':'CMS footnote code(s)',
 'period_start':'start of data window (ISO date)',
 'period_end':'end of data window (ISO date)',
 'ed_volume_tier':'ED-volume tier of a state ED-time value (low/medium/high/very high); blank otherwise',
 'statistic':'score | avg | median | n_better | n_same | n_worse | n_too_few',
 'value':'numeric value of the statistic; blank if not reported',
 'value_status':'"reported" or the text CMS shows instead of a number'}
schema = pd.DataFrame([('facility', c, DESC[c]) for c in FACILITY_COLS] + [('state', c, DESC[c]) for c in STATE_COLS], columns=['format','column','description'])
schema.to_csv(OUT_DIR / "standard_schema.csv", index=False)
measures = (pd.concat([fac_cd, fac_te, fac_asc])[['measure_group','measure_id','measure_short_name','measure_name','better_direction']]
            .drop_duplicates('measure_id').reset_index(drop=True))
measures.to_csv(OUT_DIR / "measure_lookup.csv", index=False)
print(len(schema), "schema rows;", len(measures), "measures")

46 schema rows; 62 measures
